In [0]:
%run ../../Configurations/Init_Scripts

In [0]:

dbutils.widgets.text("CatalogName", "cd_prod")
CatalogName = dbutils.widgets.get("CatalogName")


dbutils.widgets.text("ExternalLocationName", "mntprod")
ExternalLocationName = dbutils.widgets.get("ExternalLocationName")
filePrefix = '/dbfs/'+ExternalLocationName

dbutils.widgets.text('ExternalLocation_raw',"/mntprod_raw")
ExternalLocation_raw = dbutils.widgets.get('ExternalLocation_raw')
input_path=ExternalLocation_raw+'/MasterData/Comments'

dbutils.widgets.text("ExternalLocationName_silver", "/mntprod_silver")
ExternalLocationName_silver = dbutils.widgets.get("ExternalLocationName_silver")
output_path=ExternalLocationName_silver+'/DIMComments'

In [0]:
spark.conf.set("spark.sql.catalog.current", CatalogName)

In [0]:
# input_path = '/mnt/raw/MasterData/Comments'
# output_path = '/mnt/silver/DIMComments'


In [0]:
df_Comments = spark.read.option("delimiter", "~").csv(input_path,header=True)

df_Comments = (df_Comments.withColumn('CreatedBy',lit('ADB_Promotion_InitialLoad'))
                          .withColumn('CreatedDate',current_timestamp())
                          .withColumn('UpdatedBy',lit('ADB_Promotion_InitialLoad'))
                          .withColumn('UpdatedDate',current_timestamp()))

deltaTable = DeltaTable.forPath(spark, output_path)

deltaTable.alias("tgt").merge(  
    df_Comments.alias("src"),
    "tgt.CommentId = src.CommentId"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

In [0]:
display(df_Comments)

In [0]:
# DLTableName = 'Promotion.DIM_Comments'
# SilverFilePath = '/mnt/silver/DIMComments'
# spark.sql("DROP TABLE IF EXISTS {0}".format(DLTableName))
# spark.sql("CREATE TABLE IF NOT EXISTS {0} USING DELTA LOCATION '{1}'".format(DLTableName,SilverFilePath))